# HCG against MD



## Setup

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import MDAnalysis as mda
import mdtraj as md
import numpy as np
import seaborn as sns
from MDAnalysis.analysis.dihedrals import Ramachandran

path_data = "/home/tb/adlouet/DATA/PTM_LIBRARY/post_processed_trajectories_2025/"
path_hcg = path_data + "HCG_output/"
path_md = path_data + "single_chain/"

systems = {'full_phosph': ('phosphorylated/full_phosp', 'phosphorylation/full_phosph/rep_0'),
           'unmodified': ('unmodified/unmodified', 'unmodified/unmodified/rep_0')}
panel_labels = {'full_phosph': 'All Ser/Thr', 'unmodified': 'Unmodified'}
method_colors = {'MD': '#377eb8', 'HCG': '#1b9e77'}

fig_dir = 'figures_hcg'
cache_dir = 'hcg_cache' 
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(cache_dir, exist_ok=True)

full_width = 7.0
sns.set_theme(style="ticks", context="paper", font="Liberation Sans", rc={
    'font.size': 10, 'axes.labelsize': 11, 'axes.titlesize': 11, 'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'legend.fontsize': 10, 'axes.linewidth': 0.8, 'xtick.major.width': 0.8, 'ytick.major.width': 0.8,
    'mathtext.default': 'regular', 'pdf.fonttype': 42, 'ps.fonttype': 42,
})

def save_figure(fig, name):
    for ext in ['pdf', 'png']:
        fig.savefig(f'{fig_dir}/{name}.{ext}', dpi=600, bbox_inches=fig.bbox_inches)
    plt.show()

def cached(name, compute):
    path = f'{cache_dir}/{name}.pck'
    if os.path.exists(path):
        with open(path, 'rb') as file:
            return pickle.load(file)
    result = compute()
    with open(path, 'wb') as file:
        pickle.dump(result, file)
    return result

def trajectory_files(y, method):
    hcg_dir, md_dir = systems[y]
    if method == 'HCG':
        return path_hcg + hcg_dir + '/pair.xtc', path_hcg + hcg_dir + '/pair0.pdb'
    return path_md + md_dir + '/prod/pbc_nojump_2025.xtc', path_md + md_dir + '/box/charmm_box.gro'

def plot_hcg_comparison(values, xlabel):
    fig, axes = plt.subplots(1, 2, figsize=(full_width, 3.1), sharex=True, sharey=True, constrained_layout=True)
    for ax, y in zip(axes, systems):
        for method, color in method_colors.items():
            sns.kdeplot(x=values(y, method), label=method, color=color, fill=True, linewidth=1.0, ax=ax)
        ax.set_title(panel_labels[y])
        ax.set_xlabel(xlabel)
        sns.despine(ax=ax)
    axes[0].set_ylabel('Probability density')
    fig.legend(*axes[0].get_legend_handles_labels(), loc='outside lower center', ncol=2, frameon=False,
               columnspacing=1.0, handlelength=1.4, fontsize=9.5)
    return fig, axes

## Figure S1a: radius of gyration distributions

In [ ]:
def compute_rg(y, method):
    xtc, top = trajectory_files(y, method)
    return md.compute_rg(md.load(xtc, top=top), masses=None) * 10  # nm to A

fig, axes = plot_hcg_comparison(lambda y, method: cached(f'rg_{y}_{method}', lambda: compute_rg(y, method)),
                                r'$R_G$ ($\AA$)')
axes[0].set_xlim(10, 100)
save_figure(fig, 'figure_s1a_rg')

## Figure S1b: ψ dihedral distributions

In [ ]:
def compute_psi(y, method):
    xtc, top = trajectory_files(y, method)
    universe = mda.Universe(top, xtc)
    return Ramachandran(universe.select_atoms('protein')).run().angles[:, :, 1].ravel()

fig, axes = plot_hcg_comparison(lambda y, method: cached(f'psi_{y}_{method}', lambda: compute_psi(y, method)),
                                r'$\psi$ (degrees)')
axes[0].set_xlim(-180, 180)
axes[0].set_xticks([-180, -90, 0, 90, 180])
save_figure(fig, 'figure_s1b_psi')